In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
samples = list(np.loadtxt(f'{projdir}/txt/samples.txt', dtype=str))
s = samples[0]

In [ ]:
!ls -lt {projdir}/h5ad/atac/processed/archr/archr_gene_activity.h5ad

In [ ]:
atac = sc.read_h5ad(f'{projdir}/h5ad/atac/processed/archr/intersect_rna.h5ad')

In [ ]:
atac.var.index = atac.var['name']

In [ ]:
atac.var

### add trajectory score

In [ ]:
traj_genes = pd.read_csv(f'{projdir}/csv/trajectory/IPS.Trajectory.Gene.List.txt', sep='/t', header=0, index_col=0)
traj_genes.shape

In [ ]:
traj_genes.head()

In [ ]:
esc_genes = traj_genes[traj_genes['Class'] == 'ESC']
fibro_genes = traj_genes[traj_genes['Class'] == 'Fibro']
esc_genes.shape, fibro_genes.shape, \
esc_genes.shape[0] + fibro_genes.shape[0], traj_genes.shape

In [ ]:
atac.var['gene_symbol'] = [x.split('.')[0] for x in atac.var.index]
atac.var['gene_symbol'].head()

In [ ]:
esc_mask = atac.var['gene_symbol'].isin(esc_genes.index)
np.sum(esc_mask), esc_genes.index.shape

In [ ]:
np.sum(atac[:, esc_mask].X, axis=1).shape

In [ ]:
def score(x):
    return  x - np.min(x) / (np.max(x) - np.min(x))

def get_traj_score(adata, traj_genes):
    fibro_mask = adata.var.index.isin(fibro_genes.index)
    esc_mask = adata.var.index.isin(esc_genes.index)
    esc_scores = score(np.mean(adata[:, esc_mask].X, axis=1))
    fibro_scores = score(np.mean(adata[:, fibro_mask].X, axis=1))
    return esc_scores - fibro_scores

In [ ]:
atac.var

In [ ]:
traj_scores = get_traj_score(atac, traj_genes)
atac.obs['traj_score'] = traj_scores.copy()

In [ ]:
atac.obs[['traj_score']].to_csv(f'{projdir}/csv/atac/filtered_traj_score.csv', sep='\t', header=True, index=True)

In [ ]:
!date